# Kuramoto Dynamics

A Kuramoto oscillator{footcite}`kuramoto1975` is a single phase $\theta_i$ turning on a circle at its own natural frequency $\omega_i$.
Couple a population of them on a graph and above a critical coupling they abandon their private frequencies and turn as one.

In [ ]:
# Install pyGD first when this notebook runs somewhere it is not installed,
# such as Google Colab. Where pyGD is already installed this does nothing.
try:
    import pyGD
except ImportError:
    %pip install --quiet git+https://github.com/EternalTime/pyGD.git

## Driving the class

[`Kuramoto`](#pyGD.dynamics.Kuramoto) takes a coupling `sigma`, a {mod}`networkx` graph, and a vector of natural frequencies; it pulls the adjacency out once as a sparse matrix and steps the phases with an explicit Euler update.
`evolve` advances one step, `run` advances many, and `update_order_parameter` refreshes the collective quantities:

In [ ]:
import numpy as np
import networkx as nx
from pyGD import Kuramoto

rng = np.random.default_rng(1)
G = nx.erdos_renyi_graph(400, 0.03, seed=1)
omegas = rng.standard_normal(G.number_of_nodes())

env = Kuramoto(sigma=1.0, G=G, omegas=omegas, rng=rng)
env.run(500)                 # transient, unmeasured
r_hist = env.run(200, record=True)
print(r_hist.mean())         # 0.9958

Writing $Z = r\,e^{i\phi}$ for the population average of $e^{i\theta}$, the magnitude $r$ measures how tightly the phases bunch and $\phi$ the mean phase they bunch around.
Both live on the object as `env.r`, `env.phi`, and `env.Z` once `update_order_parameter` has run.

## How far you can push the coupling

The update is explicit Euler, so `dt` is not a free parameter.
Linearize the step about a phase-locked state and the coupling term becomes the graph Laplacian $L$, giving $\theta_{n+1} = (I - dt\,\sigma L)\,\theta_n$, so the step is stable only while

$$
dt\,\sigma\,\lambda_{\max}(L) \lesssim 2,
$$

where $\lambda_{\max}(L)$ is the largest Laplacian eigenvalue.
Past that the stiffest modes overshoot every step and `r` *falls* as you raise the coupling, an artifact that looks exactly like physics.

The maximum degree is only a proxy for that eigenvalue: $k_{\max} + 1 \le \lambda_{\max}(L) \le 2 k_{\max}$.
On an Erdős–Rényi graph $\lambda_{\max}$ sits near the lower end, but on a near-regular or bipartite graph it approaches $2 k_{\max}$, where a rule written in $k_{\max}$ alone is optimistic by up to a factor of two.
Compute the eigenvalue when the margin matters:

In [ ]:
lam_max = np.linalg.eigvalsh(nx.laplacian_matrix(G).toarray())[-1]

The numbers below come from `nx.erdos_renyi_graph(500, 0.02, seed=0)` with `omegas = np.random.default_rng(0).standard_normal(500)` and the environment seeded `np.random.default_rng(0)`, a sparser graph than this page's running example.
It has $k_{\max} = 19$ but $\lambda_{\max}(L) = 21.89$, so the bound is crossed at $\sigma = 1.46$ rather than the $\sigma = 1.68$ you get by putting $k_{\max}$ itself in place of the eigenvalue.
Each row is 600 steps at the default `dt = 1/16` against 4800 steps at `dt = 1/128`, both averaged over the same physical window $t \in [31.25, 37.5]$, the last 100 coarse steps and the last 800 fine ones:

| `sigma` | $\sigma \lambda_{\max}(L)\, dt$ | `r` at `dt = 1/16` | `r` converged |
| --- | --- | --- | --- |
| 1.0 | 1.37 | 0.9887 | 0.9887 |
| 1.5 | 2.05 | 0.9944 | 0.9951 |
| 2.0 | 2.74 | 0.9415 | 0.9972 |

The bound is a warning threshold, not a cliff.
`sigma = 1.5` already sits just past it at 2.05 and the error is still only 0.0006; it becomes visible just after the crossing and then grows fast, reaching 0.056 by `sigma = 2.0`.
That last row is the trap: `r = 0.94` looks like a plausible partially-locked state, but the true answer is 0.997 and the missing 0.06 is integration error.
If you need a larger `sigma`, shrink `dt` to match, and remember that `Yokai`'s `speed` is per environment step, so halving `dt` also doubles how fast the agent hops in physical time.

## The synchronization transition

The single number worth plotting first is $r$ against coupling.
Below a critical $\sigma_c$ the oscillators ignore each other and $r$ sits near zero; above it, an extensive fraction locks and $r$ climbs toward one{footcite}`strogatz2000,acebron2005`.
Sweep the coupling and watch the knee appear:

In [ ]:
import numpy as np
import networkx as nx
from pyGD import Kuramoto

G = nx.erdos_renyi_graph(400, 0.05, seed=2)
sigmas = np.linspace(0, 0.8, 24)

curve = []
for sigma in sigmas:
    rng = np.random.default_rng(0)
    omegas = rng.standard_normal(G.number_of_nodes())
    env = Kuramoto(sigma, G, omegas, rng=rng)
    env.run(500)
    curve.append(env.run(200, record=True).mean())

Plot `curve` against `sigmas` and you have the shape every figure in the paper{footcite}`sowinski2024information` is built on.
Convince yourself the knee moves with the graph's density: a sparser graph needs stronger coupling to lock{footcite}`rodrigues2016`.

## The coarse-grained limit

[`KuramotoCG`](#pyGD.dynamics.KuramotoCG) is the same environment with the agent folded in.
Rather than simulate a demon hopping and kicking, it adds the two terms the demon leaves behind in the continuum limit - a degree-weighted shift of the natural frequencies and a Wiener fluctuation whose amplitude grows with degree - so hubs sit in hotter baths than leaves.
The agent's two parameters survive only through their product `ab` $= \alpha\beta$:

In [ ]:
from pyGD import KuramotoCG

rng = np.random.default_rng(3)
env = KuramotoCG(sigma=0.4, G=G, omegas=rng.standard_normal(G.number_of_nodes()),
                 ab=0.08, rng=rng)
env.run(500)
print(env.run(200, record=True).mean())

The two dynamics stay independent classes, exactly as the MATLAB has them; the derivation that turns one into the other is sketched in [](theory.rst) and given in full in the paper.
To watch the demon rather than its shadow, go to [](guide_yokai.ipynb).

## References

```{footbibliography}
```